# Network Intrusion Detection System (NIDS)
1. Load and explore the data
2. EDA with 8 charts
3. Preprocessing
4. Feature Engineering
5. Train models (Random Forest + Logistic Regression)
6. Evaluate results

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## Step 2: Load the Data

In [ ]:
train = pd.read_csv('archive/UNSW_NB15_training-set.csv')
test  = pd.read_csv('archive/UNSW_NB15_testing-set.csv')

train.shape
test.shape

## Step 3: Quick Data Overview

In [ ]:
train.head()

In [ ]:
train.dtypes

In [ ]:
missing = train.isnull().sum()


In [ ]:
train.describe()

## Step 4: Exploratory Data Analysis (EDA)



### Chart 1: Normal vs Attack (Label Distribution)

In [ ]:
label_counts = train['label'].value_counts().sort_index()
plt.figure(figsize=(7, 5))
bars = plt.bar(['Normal (0)', 'Attack (1)'], label_counts.values,color=['steelblue', 'tomato'], width=0.4)
plt.title('Normal vs Attack Traffic', fontsize=14)
plt.ylabel('Number of Records')
plt.tight_layout()
plt.show()

### Chart 2: Attack Category Distribution

In [ ]:
attack_counts = train['attack_cat'].value_counts()
plt.figure(figsize=(11, 5))
sns.barplot(x=attack_counts.index, y=attack_counts.values, palette='viridis')
plt.title('Number of Records per Attack Category', fontsize=14)
plt.xlabel('Attack Category')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### Chart 3: Protocol Distribution

In [ ]:
top_protos = train['proto'].value_counts().head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_protos.index, y=top_protos.values, palette='coolwarm')
plt.title('Top 10 Network Protocols', fontsize=14)
plt.xlabel('Protocol')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Chart 4: Service Distribution

In [ ]:
top_services = train['service'].value_counts().head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_services.index, y=top_services.values, palette='magma')
plt.title('Top 10 Network Services', fontsize=14)
plt.xlabel('Service')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Chart 5: Packet Count Distribution (Source vs Destination)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
spkts_cap = train['spkts'].clip(upper=train['spkts'].quantile(0.99))
dpkts_cap = train['dpkts'].clip(upper=train['dpkts'].quantile(0.99))
axes[0].hist(spkts_cap, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Source Packet Count')
axes[0].set_xlabel('Packets')
axes[0].set_ylabel('Frequency')
axes[1].hist(dpkts_cap, bins=40, color='salmon', edgecolor='white')
axes[1].set_title('Destination Packet Count')
axes[1].set_xlabel('Packets')
plt.suptitle('Packet Count Distributions (capped at 99th percentile)', fontsize=13)
plt.tight_layout()
plt.show()

### Chart 6: Connection Duration: Normal vs Attack

In [ ]:
dur_cap = train['dur'].clip(upper=train['dur'].quantile(0.99))
plt.figure(figsize=(10, 5))
plt.hist(dur_cap[train['label'] == 0], bins=50, alpha=0.6, color='steelblue', label='Normal')
plt.hist(dur_cap[train['label'] == 1], bins=50, alpha=0.6, color='tomato',    label='Attack')
plt.title('Connection Duration: Normal vs Attack', fontsize=14)
plt.xlabel('Duration (seconds)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

### Chart 7: Correlation Heatmap

In [ ]:
numeric_cols = ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sload', 'dload', 'sjit', 'djit', 'label']
corr = train[numeric_cols].corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',linewidths=0.5, square=True)
plt.title('Correlation Heatmap (selected features)', fontsize=14)
plt.show()

### Chart 8: Source Bytes vs Destination Bytes (coloured by label)

In [ ]:
sample = train.sample(3000, random_state=42)
sns.scatterplot(data=sample,x=sample['sbytes'].clip(upper=sample['sbytes'].quantile(0.99)),
    y=sample['dbytes'].clip(upper=sample['dbytes'].quantile(0.99)),
    hue='label', palette={0: 'steelblue', 1: 'tomato'}, alpha=0.4, s=15
)
plt.title('Source Bytes vs Destination Bytes (3000 sample)')
plt.show()


## Step 5: Preprocessing


In [ ]:
df_train = train.copy()
df_test  = test.copy()
df_train.drop(columns=['id', 'attack_cat'], inplace=True) # 'id' is just a row number, not useful for the model
df_test.drop(columns=['id', 'attack_cat'],  inplace=True) # 'attack_cat' leaks the answer (label is our actual target)
print('Columns after dropping:', df_train.shape[1])

In [ ]:
text_cols = df_train.select_dtypes(include='object').columns.tolist()
print('Text columns to encode:', text_cols)

In [ ]:
# LabelEncoder converts each unique text value to an integer
# e.g. 'tcp' -> 0, 'udp' -> 1
le = LabelEncoder()

for col in text_cols:
    df_train[col] = le.fit_transform(df_train[col].astype(str))
    known = list(le.classes_)
    df_test[col] = df_test[col].astype(str).apply(
        lambda x: le.transform([x])[0] if x in known else -1
    )
print('Encoding done!')

In [ ]:
X_train = df_train.drop(columns=['label'])
y_train = df_train['label']
X_test = df_test.drop(columns=['label'])
y_test = df_test['label']
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling done!')

## Step 6: Feature Engineering

In [ ]:
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test  = pd.DataFrame(X_test_scaled,  columns=X_test.columns)

# total traffic = source + destination bytes
X_train['total_bytes'] = X_train['sbytes'] + X_train['dbytes']
X_test['total_bytes']  = X_test['sbytes']  + X_test['dbytes']

# total packets
X_train['total_pkts'] = X_train['spkts'] + X_train['dpkts']
X_test['total_pkts']  = X_test['spkts']  + X_test['dpkts']

# average bytes per packet
X_train['bytes_per_pkt'] = X_train['total_bytes'] / (X_train['total_pkts'] + 1e-6)
X_test['bytes_per_pkt']  = X_test['total_bytes']  / (X_test['total_pkts']  + 1e-6)

# TTL difference between source and destination
X_train['ttl_diff'] = abs(X_train['sttl'] - X_train['dttl'])
X_test['ttl_diff']  = abs(X_test['sttl']  - X_test['dttl'])


## Step 7:  Train Models

In [ ]:
print('Training Random Forest')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)


In [ ]:
print('Training Logistic Regression')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)


## Step 8: Evaluate the Models

In [ ]:
rf_preds = rf.predict(X_test)
lr_preds = lr.predict(X_test)

print('Random Forest Accuracy:', round(accuracy_score(y_test, rf_preds) * 100, 2),'%')
print('Logistic Regression Accuracy:', round(accuracy_score(y_test, lr_preds) * 100, 2),'%')

In [ ]:
print('=== Random Forest ===' )
print(classification_report(y_test, rf_preds, target_names=['Normal', 'Attack']))

In [ ]:
print('=== Logistic Regression ===')
print(classification_report(y_test, lr_preds, target_names=['Normal', 'Attack']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, preds, title, cmap in zip(
    axes,
    [rf_preds, lr_preds],
    ['Random Forest', 'Logistic Regression'],
    ['Blues', 'Oranges']
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Pred Normal', 'Pred Attack'],
                yticklabels=['Actual Normal', 'Actual Attack'])
    ax.set_title(title + ' — Confusion Matrix', fontsize=12)

plt.tight_layout()
plt.show()

## Step 9: Feature Importance (Random Forest)

In [ ]:
feat_df = pd.DataFrame({
    'feature':    X_train.columns,
    'importance': rf.feature_importances_
})
feat_df = feat_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feat_df, palette='viridis')
plt.title('Top 15 Most Important Features (Random Forest)', fontsize=14)
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()